# ⛔ VOID — RETRACTED DESIGN (R2/R3). DO NOT RUN.This notebook drives the **v1 partner-selection gate** (`PRIMARY GATE: additive-frozen conflict PMR >= 0.60`, `CALL: GO`). That gate was **retracted in full**:- an **oracle site detector** scores PMR **1.000** on the conflict set — the task never required partner information;- a **geometry oracle reading zero nucleotides** also scores **1.000**, because the acceptor was pinned to a constant position;- a fair PWM scores **0.687** vs 0.392 chance.See `studies/splicebert_partner_selection/CLAIMS.md` **R2/R3**. The live science is the **coupling** study — `FINDINGS.md`.

# SpliceBERT partner-selection — GPU validation matrix (Colab)

Runs the reduced-scale (or full) behavioral validation matrix from the SpliceBERT
circuit-discovery scaffold on a CUDA GPU. **Validation-only — the held-out test
split is never opened here.**

**Parallelize across runtimes:** open this notebook in *N* separate Colab runtimes.
In each, set `NUM_SHARDS = N` and a distinct `SHARD_ID` (0..N-1). Each runtime runs
its slice of the 25 jobs and writes results to the same Google Drive folder; the
final cell aggregates all shards.

**What you need:** the `splicebert_code.zip` provided alongside this notebook
(~450 KB — code only; the ~4 GB of GENCODE + model data is re-downloaded here from
the original public sources).

Steps: GPU check → config → mount Drive → upload code → install → download data →
build dataset → baselines → **benchmark your GPU (real ETA)** → run shard → aggregate.


## 1. Check the GPU

In [ ]:
import subprocess, torch
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total,driver_version",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())
assert torch.cuda.is_available(), "No GPU! Use Runtime > Change runtime type > GPU (T4/L4/A100)."
print("torch", torch.__version__, "| CUDA", torch.version.cuda, "| GPU", torch.cuda.get_device_name(0))

## 2. Configuration

Edit for your run. To **parallelize**, launch several runtimes with the same
`NUM_SHARDS` and different `SHARD_ID`. `MAX_TRAIN=12000` matches the reduced-scale
scope chosen for the local run (keeps results comparable); set `MAX_TRAIN=None` for
the full 177k-example training set (a GPU can afford it — check the benchmark cell).

In [ ]:
# ==== EDIT ME ====
NUM_SHARDS = 1          # number of parallel Colab runtimes you will launch
SHARD_ID   = 0          # 0 .. NUM_SHARDS-1  (unique per runtime)

SEEDS   = [0, 1, 2, 3, 4]
REGIMES = ["additive-frozen", "additive-random-frozen", "interaction-frozen",
           "additive-finetuned", "interaction-finetuned"]

MAX_TRAIN = 12000       # None = full 177,418. Reduced-scale default = comparable to local run.
MAX_VAL   = 4000
EVAL_MAX  = 4000

USE_DRIVE = True        # persist outputs to Drive (REQUIRED to collect multiple shards)
GENCODE_RELEASE = 50
# =================
print(f"shard {SHARD_ID}/{NUM_SHARDS} | seeds {SEEDS} | regimes {len(REGIMES)} | "
      f"MAX_TRAIN={MAX_TRAIN}")

## 3. Mount Google Drive (shared output store)

In [ ]:
import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = "/content/drive/MyDrive/splicebert_run"
else:
    DRIVE_ROOT = "/content/splicebert_run"
os.makedirs(DRIVE_ROOT, exist_ok=True)
print("outputs ->", DRIVE_ROOT)

## 4. Upload the code zip

Run the cell and pick `splicebert_code.zip`. (Re-run-safe: skips if already unpacked.)
To avoid re-uploading on every reconnect, you can instead drop the zip in
`MyDrive/splicebert_run/` — this cell will find it there automatically.

In [ ]:
import os, zipfile, shutil
WORK = "/content/work"; os.makedirs(WORK, exist_ok=True)
REPO = os.path.join(WORK, "gfm_splicebert_implementation")
if not os.path.exists(os.path.join(REPO, "pyproject.toml")):
    zpath = None
    drive_zip = os.path.join(DRIVE_ROOT, "splicebert_code.zip")
    if os.path.exists(drive_zip):
        zpath = drive_zip; print("found code zip on Drive")
    else:
        from google.colab import files
        print("Upload splicebert_code.zip:")
        up = files.upload(); zpath = list(up)[0]
    os.makedirs(REPO, exist_ok=True)
    with zipfile.ZipFile(zpath) as z: z.extractall(REPO)
print("repo:", REPO, "|", sorted(os.listdir(REPO))[:8])

## 5. Install dependencies

In [ ]:
import sys, os, importlib
# Colab already ships a CUDA torch build; only add the study deps.
!pip install -q "transformers>=4.40" "pyfaidx>=0.8" scipy 2>&1 | tail -3
SRC = os.path.join(REPO, "src"); sys.path.insert(0, SRC)
os.environ["PYTHONPATH"] = SRC + os.pathsep + os.environ.get("PYTHONPATH", "")
importlib.invalidate_caches()
import gfmcd.adapters.splicebert  # noqa
import transformers; print("gfmcd import OK | transformers", transformers.__version__)

## 6. Download GENCODE + SpliceBERT (once)

~1 GB download, ~3 GB decompressed FASTA. Cached to Drive so later runtimes/reconnects
skip it. Sets `model.path` to the official **SpliceBERT.1024nt** checkpoint.

In [ ]:
import subprocess, json, os, shutil
os.chdir(REPO)
cfgp = "configs/splicebert_partner_selection/base.json"
RAW_LOCAL = "data"
RAW_DRIVE = os.path.join(DRIVE_ROOT, "raw_data")   # optional cache of extracted model + gencode

need_fetch = not (os.path.exists("data/external/gencode/v%d/GRCh38.primary_assembly.genome.fa" % GENCODE_RELEASE)
                  and os.path.exists("data/models/splicebert_official_archive/extracted/models/SpliceBERT.1024nt/pytorch_model.bin"))
if need_fetch and os.path.isdir(os.path.join(RAW_DRIVE, "external")):
    print("restoring raw data from Drive cache ...")
    shutil.copytree(RAW_DRIVE, RAW_LOCAL, dirs_exist_ok=True); need_fetch = False
if need_fetch:
    subprocess.run(["bash", "scripts/splicebert/fetch_resources.sh", str(GENCODE_RELEASE)], check=True)
    # cache to Drive (model archive extracted + gencode) for reuse; skip the giant .fa.gz to save space
    try:
        os.makedirs(RAW_DRIVE, exist_ok=True)
        shutil.copytree("data/models", os.path.join(RAW_DRIVE, "models"), dirs_exist_ok=True)
        shutil.copytree("data/external", os.path.join(RAW_DRIVE, "external"), dirs_exist_ok=True)
        print("cached raw data to Drive")
    except Exception as e:
        print("Drive cache skipped:", e)

cfg = json.load(open(cfgp))
cfg["model"]["path"] = "data/models/splicebert_official_archive/extracted/models/SpliceBERT.1024nt"
json.dump(cfg, open(cfgp, "w"), indent=2)
print("model.path =", cfg["model"]["path"])

MODEL_DIR = cfg["model"]["path"]; TGZ = "data/models/splicebert_official_archive/models.tar.gz"
def ensure_weights():
    if not os.path.exists(os.path.join(MODEL_DIR, "pytorch_model.bin")):
        print("weights missing -> re-extracting"); subprocess.run(
            ["tar","-xzf",TGZ,"-C","data/models/splicebert_official_archive/extracted"], check=True)
ensure_weights()

## 7. Build the exon-pair dataset (once, cached to Drive)

Deterministic (chromosome split, no randomness) so every runtime produces the identical
dataset + conflict set. ~15 min the first time; restored from Drive afterwards.

In [ ]:
import os, shutil, subprocess
DS_LOCAL = "data/processed/splicebert_partner_selection/v1"
DS_DRIVE = os.path.join(DRIVE_ROOT, "dataset_v1")
if os.path.exists(os.path.join(DS_DRIVE, "train.jsonl")):
    os.makedirs(DS_LOCAL, exist_ok=True)
    for f in os.listdir(DS_DRIVE): shutil.copy(os.path.join(DS_DRIVE, f), DS_LOCAL)
    print("dataset restored from Drive")
else:
    subprocess.run(["python", "scripts/splicebert/build_dataset.py", "--config", cfgp], check=True)
    os.makedirs(DS_DRIVE, exist_ok=True)
    for f in os.listdir(DS_LOCAL):
        if f.endswith((".jsonl", ".json")): shutil.copy(os.path.join(DS_LOCAL, f), DS_DRIVE)
    print("dataset built + cached to Drive")
import subprocess as sp
print(sp.run(["wc","-l",DS_LOCAL+"/train.jsonl",DS_LOCAL+"/val.jsonl",DS_LOCAL+"/test.jsonl"],
             capture_output=True,text=True).stdout)

## 8. Local baseline + native masked-LM dependency (once, cached)

Freezes the PWM+length shortcut baseline and the natural conflict set (the gate's
comparison target), and runs the zero-shot dependency pilot.

In [ ]:
import os, shutil, subprocess
BASE_LOCAL = "results/splicebert_partner_selection/baselines"
BASE_DRIVE = os.path.join(DRIVE_ROOT, "baselines")
if os.path.exists(os.path.join(BASE_DRIVE, "val_local_conflicts.jsonl")):
    os.makedirs(BASE_LOCAL, exist_ok=True)
    for f in os.listdir(BASE_DRIVE): shutil.copy(os.path.join(BASE_DRIVE, f), BASE_LOCAL)
    print("baseline restored from Drive")
else:
    subprocess.run(["python","scripts/splicebert/evaluate_local_baselines.py","--config",cfgp], check=True)
    subprocess.run(["python","scripts/splicebert/evaluate_native_dependency.py","--config",cfgp,
                    "--device","cuda","--local-baseline",BASE_LOCAL+"/local_pair_baseline.pkl"], check=False)
    os.makedirs(BASE_DRIVE, exist_ok=True)
    for f in os.listdir(BASE_LOCAL): shutil.copy(os.path.join(BASE_LOCAL, f), BASE_DRIVE)
    print("baseline fit + cached to Drive")
import json
print("local baseline:", json.load(open(BASE_LOCAL+"/metrics.json"))["val"])

## 9. Benchmark *your* GPU → real ETA

Times a frozen forward and a fine-tuned forward+backward on real batches, then projects
the wall-clock for **your shard** and the **whole matrix** with the current config.

In [ ]:
import time, torch
from gfmcd.adapters.splicebert import SpliceBertAdapter
from gfmcd.splicing.io import load_jsonl
from gfmcd.splicing.readout import AdditivePairReadout, pair_cross_entropy
from gfmcd.splicing.training import forward_examples
ensure_weights()
train = load_jsonl(DS_LOCAL + "/train.jsonl")
a = SpliceBertAdapter.from_pretrained(MODEL_DIR, local_files_only=True, device="cuda")
ro = AdditivePairReadout(a.hidden_size, site_hidden=128, length_hidden=16).to(a.device)
batch = train[:8]
def bench(grad, n=8):
    for p in a.model.parameters(): p.requires_grad_(grad)
    for _ in range(3):
        l, pb = forward_examples(a, ro, batch, grad_backbone=grad)
        if grad: pair_cross_entropy(l, pb).backward()
    torch.cuda.synchronize(); t = time.time()
    for _ in range(n):
        l, pb = forward_examples(a, ro, batch, grad_backbone=grad)
        if grad: pair_cross_entropy(l, pb).backward()
    torch.cuda.synchronize(); return (time.time()-t)/n
frozen_s = bench(False); ft_s = bench(True)
print(f"frozen fwd/batch = {frozen_s*1000:.0f} ms | finetuned fwd+bwd/batch = {ft_s*1000:.0f} ms")

n_train = (MAX_TRAIN or 177418); n_val = (MAX_VAL or 17069)
bpe_tr = -(-n_train//8); bpe_val = -(-min(n_val,EVAL_MAX or n_val)//8)
EPOCHS = 7  # typical after early stopping (cap 10)
def run_min(is_ft):
    tr = bpe_tr*(ft_s if is_ft else frozen_s)*EPOCHS
    val = bpe_val*frozen_s*EPOCHS                      # eval is frozen fwd
    evals = 2*5*(-(-min((MAX_VAL or n_val),EVAL_MAX)//8))*frozen_s  # 2 evals x 5 conditions
    return (tr+val+evals)/60
frozen_run = run_min(False); ft_run = run_min(True)
jobs = [(s,r) for s in SEEDS for r in REGIMES]
my = [(s,r) for i,(s,r) in enumerate(jobs) if i % NUM_SHARDS == SHARD_ID]
def cost(r): return ft_run if r.endswith("finetuned") else frozen_run
print(f"~per frozen run {frozen_run:.0f} min | ~per finetuned run {ft_run:.0f} min")
print(f"THIS SHARD: {len(my)} jobs -> ~{sum(cost(r) for _,r in my)/60:.1f} h")
print(f"WHOLE MATRIX (all shards, wall-clock if run in parallel): "
      f"~{max((sum(cost(r) for i,(s,r) in enumerate(jobs) if i%NUM_SHARDS==k)/60) for k in range(NUM_SHARDS)):.1f} h")
del a, ro; torch.cuda.empty_cache()

## 10. Run this shard

Trains + evaluates each job in the shard on the GPU and copies checkpoints + metrics to
Drive. Re-run-safe: skips jobs already completed on Drive. `additive-frozen` is the
registered primary condition; the test split is never touched.

In [ ]:
import os, json, time, shutil, subprocess
RUNS_DRIVE = os.path.join(DRIVE_ROOT, "runs"); os.makedirs(RUNS_DRIVE, exist_ok=True)
CONFLICTS = BASE_LOCAL + "/val_local_conflicts.jsonl"
jobs = [(s,r) for s in SEEDS for r in REGIMES]
my = [(s,r) for i,(s,r) in enumerate(jobs) if i % NUM_SHARDS == SHARD_ID]
print(f"shard {SHARD_ID}/{NUM_SHARDS}: {len(my)} jobs")

def run(cmd):
    p = subprocess.run(cmd); return p.returncode

for s, regime in my:
    tag = f"{regime}_seed{s}"; dst = os.path.join(RUNS_DRIVE, tag); os.makedirs(dst, exist_ok=True)
    if os.path.exists(os.path.join(dst, "DONE")):
        print("skip (done):", tag); continue
    ensure_weights()
    t0 = time.time()
    cmd = ["python","scripts/splicebert/train_pair_readout.py","--config",cfgp,
           "--regime",regime,"--seed",str(s),"--device","cuda"]
    if MAX_TRAIN is not None: cmd += ["--max-train",str(MAX_TRAIN)]
    if MAX_VAL   is not None: cmd += ["--max-val",str(MAX_VAL)]
    if run(cmd) != 0: print("TRAIN FAILED", tag); continue
    ckpt = f"artifacts/splicebert_partner_selection/readouts/{regime}/seed{s}/checkpoint.pt"
    for split_args, label in [([], "val"), (["--subset-file",CONFLICTS], "conflict")]:
        run(["python","scripts/splicebert/evaluate_behavior.py","--config",cfgp,
             "--checkpoint",ckpt,"--device","cuda","--max-examples",str(EVAL_MAX)] + split_args)
    # collect outputs to Drive
    rd = f"artifacts/splicebert_partner_selection/readouts/{regime}/seed{s}"
    shutil.copy(rd+"/metrics.json", dst+"/train_metrics.json")
    shutil.copy(ckpt, dst+"/checkpoint.pt")
    bdir = f"results/splicebert_partner_selection/behavior/seed{s}_{regime}"
    for sub, out in [("val/metrics.json","behavior_val.json"),
                     ("val/val_local_conflicts/metrics.json","behavior_conflict.json")]:
        p = os.path.join(bdir, sub)
        if os.path.exists(p): shutil.copy(p, os.path.join(dst, out))
    open(os.path.join(dst,"DONE"),"w").write("ok")
    print(f"DONE {tag} in {(time.time()-t0)/60:.1f} min")
print("shard complete")

## 11. Aggregate all shards → behavioral gate

Run after **every** shard shows "shard complete". Reads all runs from Drive and evaluates
the registered primary gate: additive-frozen conflict positive-margin-rate vs
additive-random-frozen, with per-seed replication. Thresholds from PREREGISTRATION.md
(conflict positive-margin-rate ≥ 0.60; replication ≥ 3/5 seeds).

In [ ]:
import os, json
RUNS_DRIVE = os.path.join(DRIVE_ROOT, "runs")
def load(tag, name):
    p = os.path.join(RUNS_DRIVE, tag, name)
    return json.load(open(p)) if os.path.exists(p) else None
rows = []
for s in SEEDS:
    rec = {"seed": s}
    for regime in REGIMES:
        b = load(f"{regime}_seed{s}", "behavior_conflict.json")
        v = load(f"{regime}_seed{s}", "behavior_val.json")
        if b: rec[regime+"_conflict_pmr"] = round(b["conditions"]["clean"]["metrics"]["positive_margin_rate"],4)
        if v: rec[regime+"_val_top1"]      = round(v["conditions"]["clean"]["metrics"]["top1_accuracy"],4)
    rows.append(rec)
import pandas as pd
df = pd.DataFrame(rows);
pd.set_option("display.width",200); pd.set_option("display.max_columns",50)
print(df.to_string(index=False))

# --- primary behavioral gate ---
af = "additive-frozen_conflict_pmr"; rf = "additive-random-frozen_conflict_pmr"
ok_seeds = [r for r in rows if af in r and rf in r and r[af] >= 0.60 and r[af] > r[rf]]
print(f"\nPRIMARY GATE: additive-frozen conflict PMR >= 0.60 AND > random-frozen")
print(f"  seeds passing: {len(ok_seeds)}/{len(SEEDS)}  (need >= 3)")
import statistics as st
vals = [r[af] for r in rows if af in r]
if vals: print(f"  additive-frozen conflict PMR: mean {st.mean(vals):.3f} range [{min(vals):.3f},{max(vals):.3f}]")
print("  CALL:", "GO (trace circuit)" if len(ok_seeds) >= 3 else "NO-GO / review")
print("\nNOTE: interaction-* and *-finetuned are secondary/diagnostic; do not use them")
print("to claim a pretrained circuit. Freeze preregistration before test unblinding.")